# Instantiating Prebuilt Agents: Practice Exercise

Build a prebuilt agent that can answer movie questions using multiple tools.

**What you'll implement:**
- Create a language model instance
- Create an agent that uses two movie lookup tools

**Estimated time:** 10 minutes

## Setup

Run this cell to load the required libraries and configure the environment.

In [2]:
# Setup - run this cell first

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_groq import ChatGroq


load_dotenv()

# Movie data for your tools to use
MOVIE_DATABASE = {
    "inception": {
        "title": "Inception",
        "director": "Christopher Nolan",
        "year": 2010,
        "rating": 8.8
    },
    "the matrix": {
        "title": "The Matrix",
        "director": "The Wachowskis",
        "year": 1999,
        "rating": 8.7
    },
    "interstellar": {
        "title": "Interstellar",
        "director": "Christopher Nolan",
        "year": 2014,
        "rating": 8.6
    }
}

# Tool 1: Look up movie by title
@tool
def lookup_movie(title: str) -> str:
    """
    Look up information about a movie by its title.
    
    Args:
        title: The name of the movie to look up
    
    Returns:
        A formatted string containing the movie's title, director, year, and rating.
        If the movie is not found, returns "Movie not found: {title}"
    """
    title_lower = title.lower()
    if title_lower in MOVIE_DATABASE:
        movie = MOVIE_DATABASE[title_lower]
        return f"{movie['title']} ({movie['year']}) - Directed by {movie['director']}, Rating: {movie['rating']}/10"
    return f"Movie not found: {title}"

# Tool 2: Look up movies by director
@tool
def lookup_movies_by_director(director: str) -> str:
    """
    Find all movies by a specific director.
    
    Args:
        director: The name of the director to search for
    
    Returns:
        A formatted string listing all movies by that director, or a message if none are found.
    """
    director_lower = director.lower()
    movies = []
    for movie_data in MOVIE_DATABASE.values():
        if director_lower in movie_data['director'].lower():
            movies.append(f"{movie_data['title']} ({movie_data['year']})")
    
    if movies:
        return f"Movies by {director}: {', '.join(movies)}"
    return f"No movies found by director: {director}"

print("Setup complete! Two tools are now available: lookup_movie and lookup_movies_by_director")

Setup complete! Two tools are now available: lookup_movie and lookup_movies_by_director


## Your Task

You are building a movie information assistant that can:
1. Look up specific movies by title
2. Find all movies by a specific director

The two tools (`lookup_movie` and `lookup_movies_by_director`) are already provided in the setup cell above.

**Your job:**
- Create a ChatOpenAI model instance
- Create an agent that uses both tools to answer movie-related questions

## Step 1: Create the Language Model

Create a ChatOpenAI model instance with the following configuration:
- Model: "gpt-4o-mini"
- Temperature: 0.1

In [3]:
# TODO: Create a ChatOpenAI model instance
model = ChatGroq(model="openai/gpt-oss-20b", temperature=0.1)

print("Model created!")

Model created!


## Step 2: Create the Agent

Create a prebuilt agent using `create_agent` that has access to both movie tools.

In [5]:
# TODO: Create the agent using create_agent
# Pass in the model and a list containing both tools: lookup_movie and lookup_movies_by_director
movie_agent = create_agent(model, tools=[lookup_movie, lookup_movies_by_director])

print("Agent created!")

Agent created!


## Run Your Implementation

Test your agent with these queries.

In [6]:
def ask_movie_agent(question: str):
    """Helper function to query the movie agent."""
    result = movie_agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })
    final_message = result["messages"][-1]
    print(final_message.content)

# Test looking up a specific movie
ask_movie_agent("Tell me about Inception")

**Inception** (2010) – Directed by Christopher Nolan  
- **Genre:** Science‑fiction, Action, Thriller  
- **Cast:** Leonardo DiCaprio, Joseph Gordon‑Levitt, Ellen Page, Tom Hardy, Ken Watanabe, Cillian Murphy, Michael Caine, and others  
- **Plot:** A skilled thief, Dom Cobb, is offered a chance to have his criminal history erased in exchange for planting an idea in someone’s mind—a process known as “inception.” He assembles a team to execute a complex, multi‑layered heist inside the subconscious of a target, navigating dream‑within‑dream levels while confronting his own memories and guilt.  
- **Reception:** The film received critical acclaim for its originality, visual effects, and complex narrative structure, earning 4 Academy Awards (Best Cinematography, Best Sound Editing, Best Sound Mixing, Best Visual Effects) and a nomination for Best Picture.  
- **Rating:** 8.8 / 10 (IMDb)


In [7]:
# Test looking up movies by director
ask_movie_agent("What movies has Christopher Nolan directed?")

Christopher Nolan has directed a number of high‑profile feature films. Here’s a quick rundown of his directorial credits (up to 2024):

| Year | Title | Notes |
|------|-------|-------|
| 2000 | **Memento** | Dark‑thriller that premiered at Cannes. |
| 2005 | **Batman Begins** | Reboot of the Batman franchise. |
| 2008 | **The Dark Knight** | Sequel to *Batman Begins*; Oscar‑nominated. |
| 2010 | **Inception** | Mind‑bending heist film. |
| 2012 | **The Dark Knight Rises** | Final chapter of the Batman trilogy. |
| 2014 | **Interstellar** | Space‑opera about time dilation. |
| 2015 | **Dunkirk** | WWII evacuation drama. |
| 2017 | **Tenet** | Time‑reverse espionage thriller. |
| 2020 | **Oppenheimer** | Biopic of J. Robert Oppenheimer (released 2023). |

If you’re looking for a specific film or want more details (cast, box‑office, awards, etc.), just let me know!
